# **06 CSV를 SQLite DB로 적재하기**

### 학습 내용
1. CSV 파일을 pandas로 읽기
2. SQLite 데이터베이스 생성 및 데이터 저장
3. 기본 SQL 쿼리 실행
4. LangChain SQLDatabase 클래스 사용법

## 0. 환경 변수 설정

In [91]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

✓ OpenAI API Key가 설정되었습니다.


## 1. CSV 파일 준비 및 확인

천안시청 조직 데이터를 사용합니다. (organizations.csv, departments.csv, office_floors.csv)

In [92]:
import pandas as pd

# 천안시청 조직 CSV 파일 로드
org_df = pd.read_csv("../datasets/organizations.csv")
dept_df = pd.read_csv("../datasets/departments.csv")
floor_df = pd.read_csv("../datasets/office_floors.csv")

print("=" * 80)
print("📋 조직 구조 (Organizations)")
print("=" * 80)
print(org_df.head())
print(f"\n총 {len(org_df)}개의 조직")

print("\n" + "=" * 80)
print("🏢 부서 정보 (Departments)")
print("=" * 80)
print(dept_df.head())
print(f"\n총 {len(dept_df)}개의 부서")

print("\n" + "=" * 80)
print("🏗️ 층별 정보 (Office Floors)")
print("=" * 80)
print(floor_df.head())
print(f"\n총 {len(floor_df)}개의 층 정보")

📋 조직 구조 (Organizations)
   org_id org_name org_type  parent_org_id  org_level org_code    description
0       1     천안시청       시청            NaN          1    CA001         천안시 본청
1       2   부시장 직속       직속            1.0          2    CA002      부시장 직속 부서
2       3    기획조정실        실            1.0          2    CA003  정책 기획 및 예산 총괄
3       4    전략산업국        국            1.0          2    CA004     산업 및 경제 정책
4       5    행정자치국        국            1.0          2    CA005     행정 및 자치 업무

총 10개의 조직

🏢 부서 정보 (Departments)
   dept_id dept_name  org_id dept_code         phone           fax  \
0      101     홍보담당관       2      D101  041-521-2080  041-521-2089   
1      102       감사관       2      D102  041-521-2040  041-521-2049   
2      103  스마트도시추진과       2      D103  041-521-2205  041-521-2209   
3      201     정책기획과       3      D201  041-521-2150  041-521-2159   
4      202     예산법무과       3      D202  041-521-2020  041-521-2028   

  floor_location         description  
0          본관 

## 2. CSV 파일 읽기 및 탐색

In [93]:
# 조직 데이터 탐색
print("📊 조직 구조 데이터:")
print(org_df.info())
print(f"\n컬럼: {list(org_df.columns)}")

print("\n📊 부서 데이터:")
print(dept_df.info())
print(f"\n컬럼: {list(dept_df.columns)}")

print("\n📊 층별 정보 데이터:")
print(floor_df.info())
print(f"\n컬럼: {list(floor_df.columns)}")

📊 조직 구조 데이터:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   org_id         10 non-null     int64  
 1   org_name       10 non-null     str    
 2   org_type       10 non-null     str    
 3   parent_org_id  9 non-null      float64
 4   org_level      10 non-null     int64  
 5   org_code       10 non-null     str    
 6   description    10 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 1.2 KB
None

컬럼: ['org_id', 'org_name', 'org_type', 'parent_org_id', 'org_level', 'org_code', 'description']

📊 부서 데이터:
<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   dept_id         45 non-null     int64
 1   dept_name       45 non-null     str  
 2   org_id          45 non-null     int64
 3   dept_code       45 

In [94]:
# 조직 타입별 통계
print("조직 타입별 개수:")
print(org_df['org_type'].value_counts())

print(org_df['org_level'].value_counts())
print("\n조직 계층별 개수:")

조직 타입별 개수:
org_type
국     7
시청    1
직속    1
실     1
Name: count, dtype: int64
org_level
2    9
1    1
Name: count, dtype: int64

조직 계층별 개수:


In [95]:
# 부서별 전화번호 확인
print("부서 정보 샘플 (부서명, 전화번호):")
print(dept_df[['dept_name', 'phone', 'floor_location']].head(10))

부서 정보 샘플 (부서명, 전화번호):
  dept_name         phone floor_location
0     홍보담당관  041-521-2080          본관 8층
1       감사관  041-521-2040          본관 4층
2  스마트도시추진과  041-521-2205          본관 7층
3     정책기획과  041-521-2150          본관 8층
4     예산법무과  041-521-2020          본관 8층
5     청년정책과  041-521-2015          본관 1층
6       세정과  041-521-2290          본관 6층
7     미래전략과  041-521-2190          본관 2층
8    일자리경제과  041-521-2170          본관 2층
9     기업지원과  041-521-2360          본관 8층


## 3. SQLite 데이터베이스 생성 및 데이터 저장

In [96]:
import sqlite3
import os
from sqlalchemy import create_engine

# SQLite 데이터베이스 파일 경로
db_path = "database/cheonan_city.db"

# 디렉토리가 없으면 생성
os.makedirs("database", exist_ok=True)

# SQLAlchemy 엔진 생성
engine = create_engine(f"sqlite:///{db_path}")

# 3개의 DataFrame을 각각 테이블로 저장
org_df.to_sql(
    name="organizations",
    con=engine,
    if_exists="replace",
    index=False
)
print(f"✓ {len(org_df)}개 행이 'organizations' 테이블에 저장되었습니다.")

dept_df.to_sql(
    name="departments",
    con=engine,
    if_exists="replace",
    index=False
)
print(f"✓ {len(dept_df)}개 행이 'departments' 테이블에 저장되었습니다.")

floor_df.to_sql(
    name="office_floors",
    con=engine,
    if_exists="replace",
    index=False
)
print(f"✓ {len(floor_df)}개 행이 'office_floors' 테이블에 저장되었습니다.")

print(f"\n✓ 데이터베이스 파일: {db_path}")

✓ 10개 행이 'organizations' 테이블에 저장되었습니다.
✓ 45개 행이 'departments' 테이블에 저장되었습니다.


✓ 12개 행이 'office_floors' 테이블에 저장되었습니다.

✓ 데이터베이스 파일: database/cheonan_city.db


## 4. 기본 SQL 쿼리 실행

### 4-1. sqlite3로 직접 쿼리

In [97]:
# SQLite 연결
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 테이블 목록 조회
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("데이터베이스의 테이블 목록:")
for table in tables:
    print(f"  - {table[0]}")

conn.close()

데이터베이스의 테이블 목록:
  - organizations
  - departments
  - office_floors


In [98]:
# 조직 데이터 조회
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("SELECT org_name, org_type, org_level FROM organizations LIMIT 5;")
rows = cursor.fetchall()

print("조직 구조 (첫 5개):")
for row in rows:
    print(f"  {row[0]} ({row[1]}, Level {row[2]})")

conn.close()

조직 구조 (첫 5개):
  천안시청 (시청, Level 1)
  부시장 직속 (직속, Level 2)
  기획조정실 (실, Level 2)
  전략산업국 (국, Level 2)
  행정자치국 (국, Level 2)


In [99]:
# 특정 조직의 부서 조회
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("SELECT dept_name, phone, floor_location FROM departments WHERE org_id = 6 ORDER BY dept_name;")
rows = cursor.fetchall()

print("복지정책국 산하 부서 (org_id=6):")
for row in rows:
    print(f"  - {row[0]}: {row[1]} ({row[2]})")

conn.close()

복지정책국 산하 부서 (org_id=6):
  - 노인복지과: 041-521-2710 (본관 1층)
  - 복지정책과: 041-521-3430 (본관 1층)
  - 아동보육과: 041-521-2690 (본관 3층)
  - 여성가족과: 041-521-2370 (본관 3층)
  - 장애인복지과: 041-521-2730 (본관 1층)


### 4-2. Pandas로 SQL 쿼리

In [100]:
# Pandas read_sql 사용 - 조직별 부서 수 집계
query = "SELECT o.org_name, COUNT(d.dept_id) as dept_count FROM organizations o LEFT JOIN departments d ON o.org_id = d.org_id GROUP BY o.org_name ORDER BY dept_count DESC;"

result_df = pd.read_sql(query, engine)

print("조직별 부서 수:")
result_df

조직별 부서 수:


,org_name,dept_count
0,문화체육국,6
1,도시주택국,6
2,농업환경국,6
3,행정자치국,5
4,전략산업국,5
5,복지정책국,5
6,건설안전교통국,5
7,기획조정실,4
8,부시장 직속,3
9,천안시청,0


## 5. LangChain SQLDatabase 사용하기

LangChain의 `SQLDatabase` 클래스는 데이터베이스와 상호작용하기 위한 편리한 인터페이스를 제공합니다.

In [101]:
from langchain_community.utilities import SQLDatabase

# SQLDatabase 객체 생성
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

print("SQLDatabase 연결 완료")
print(f"Dialect: {db.dialect}")
print(f"사용 가능한 테이블: {db.get_usable_table_names()}")

SQLDatabase 연결 완료
Dialect: sqlite
사용 가능한 테이블: ['departments', 'office_floors', 'organizations']


### 5-1. 테이블 정보 확인

In [102]:
# 테이블 스키마 확인
table_info = db.get_table_info()

print("테이블 스키마 정보:")
print(table_info)

테이블 스키마 정보:

CREATE TABLE departments (
	dept_id BIGINT, 
	dept_name TEXT, 
	org_id BIGINT, 
	dept_code TEXT, 
	phone TEXT, 
	fax TEXT, 
	floor_location TEXT, 
	description TEXT
)

/*
3 rows from departments table:
dept_id	dept_name	org_id	dept_code	phone	fax	floor_location	description
101	홍보담당관	2	D101	041-521-2080	041-521-2089	본관 8층	홍보업무 총괄 및 보도자료 관리
102	감사관	2	D102	041-521-2040	041-521-2049	본관 4층	감사업무 총괄 및 청렴윤리 관리
103	스마트도시추진과	2	D103	041-521-2205	041-521-2209	본관 7층	스마트도시 정책 및 AI산업 추진
*/


CREATE TABLE office_floors (
	floor_id BIGINT, 
	building_name TEXT, 
	floor_number BIGINT, 
	floor_name TEXT, 
	main_departments TEXT, 
	facilities TEXT, 
	area_type TEXT
)

/*
3 rows from office_floors table:
floor_id	building_name	floor_number	floor_name	main_departments	facilities	area_type
1	천안시청 본관	1	본관 1층	민원여권과,노인복지과,허가과,복지정책과,장애인복지과,청년정책과	청사카페,민원실,대기공간	민원서비스층
2	천안시청 본관	2	본관 2층	미래전략과,일자리경제과,교육청소년과,청소행정과,K-컬처박람회추진과	도솔도서관	행정문화층
3	천안시청 본관	3	본관 3층	여성가족과,아동보육과	콜센터,대회의실,천안시정공무원노동조합	복지행정층
*/


CREATE

In [103]:
# 특정 테이블의 스키마만 확인
table_info = db.get_table_info(table_names=["departments", "organizations"])
print(table_info)


CREATE TABLE departments (
	dept_id BIGINT, 
	dept_name TEXT, 
	org_id BIGINT, 
	dept_code TEXT, 
	phone TEXT, 
	fax TEXT, 
	floor_location TEXT, 
	description TEXT
)

/*
3 rows from departments table:
dept_id	dept_name	org_id	dept_code	phone	fax	floor_location	description
101	홍보담당관	2	D101	041-521-2080	041-521-2089	본관 8층	홍보업무 총괄 및 보도자료 관리
102	감사관	2	D102	041-521-2040	041-521-2049	본관 4층	감사업무 총괄 및 청렴윤리 관리
103	스마트도시추진과	2	D103	041-521-2205	041-521-2209	본관 7층	스마트도시 정책 및 AI산업 추진
*/


CREATE TABLE organizations (
	org_id BIGINT, 
	org_name TEXT, 
	org_type TEXT, 
	parent_org_id FLOAT, 
	org_level BIGINT, 
	org_code TEXT, 
	description TEXT
)

/*
3 rows from organizations table:
org_id	org_name	org_type	parent_org_id	org_level	org_code	description
1	천안시청	시청	None	1	CA001	천안시 본청
2	부시장 직속	직속	1.0	2	CA002	부시장 직속 부서
3	기획조정실	실	1.0	2	CA003	정책 기획 및 예산 총괄
*/


### 5-2. SQL 쿼리 실행

In [104]:
# 전체 부서 수
total_depts = db.run("SELECT COUNT(*) FROM departments;")
print(f"전체 부서 수: {total_depts}")

# 조직 수
total_orgs = db.run("SELECT COUNT(*) FROM organizations;")
print(f"조직 수: {total_orgs}")

# 층 수
total_floors = db.run("SELECT COUNT(*) FROM office_floors;")
print(f"층 정보 수: {total_floors}")

전체 부서 수: [(45,)]
조직 수: [(10,)]
층 정보 수: [(12,)]


## 6. 쿼리 테스트 


### 6-1. 청사 건물 층별 정보

In [105]:
# 층별 정보 쿼리
query = """
SELECT floor_number, floor_name, main_departments
FROM office_floors
WHERE building_name = '천안시청 본관'
ORDER BY floor_number;
"""
result = db.run(query)

print("본관 층별 정보:")
print(result)

본관 층별 정보:
[(1, '본관 1층', '민원여권과,노인복지과,허가과,복지정책과,장애인복지과,청년정책과'), (2, '본관 2층', '미래전략과,일자리경제과,교육청소년과,청소행정과,K-컬처박람회추진과'), (3, '본관 3층', '여성가족과,아동보육과'), (4, '본관 4층', '스마트정보과,감사관'), (5, '본관 5층', '농업정책과,환경정책과,축산과,식품안전과,기후에너지과'), (6, '본관 6층', '회계과,세정과,자치분권과,행정지원과'), (7, '본관 7층', '스마트도시추진과'), (8, '본관 8층', '홍보담당관,정책기획과,기업지원과,예산법무과'), (9, '본관 9층', '도시계획과,건설도로과,안전총괄과'), (10, '본관 10층', '대중교통과,도시재생과,교통정책과,산업단지조성추진과,하천과,체육진흥과'), (11, '본관 11층', '관광과,문화예술과,공동주택과,건축과')]


In [106]:
# 본관 1층 부서 조회
query = """
SELECT dept_name, phone
FROM departments
WHERE floor_location LIKE '%본관 1층%'
LIMIT 5;
"""
result = db.run(query)

print("본관 1층에 위치한 부서 (5개):")
print(result)

본관 1층에 위치한 부서 (5개):
[('청년정책과', '041-521-2015'), ('허가과', '041-521-2500'), ('민원여권과', '041-521-2250'), ('복지정책과', '041-521-3430'), ('노인복지과', '041-521-2710')]


In [107]:
# 층별 부서 수 및 상세 정보
query = """
SELECT
    f.floor_number,
    f.floor_name,
    f.area_type,
    COUNT(d.dept_id) as dept_count
FROM office_floors f
LEFT JOIN departments d ON d.floor_location LIKE '%' || f.floor_name || '%'
WHERE f.building_name = '천안시청 본관'
GROUP BY f.floor_number, f.floor_name, f.area_type
ORDER BY f.floor_number;
"""

result = db.run(query)

print("본관 층별 부서 배치 현황:")
print(result)

본관 층별 부서 배치 현황:
[(1, '본관 1층', '민원서비스층', 6), (2, '본관 2층', '행정문화층', 5), (3, '본관 3층', '복지행정층', 2), (4, '본관 4층', '정보관리층', 2), (5, '본관 5층', '농업환경층', 5), (6, '본관 6층', '재정행정층', 4), (7, '본관 7층', '시장실층', 1), (8, '본관 8층', '기획정책층', 4), (9, '본관 9층', '건설안전층', 3), (10, '본관 10층', '도시교통층', 6), (11, '본관 11층', '문화도시층', 4)]


### 6-2. 조직 및 부서 정보

In [108]:
# 집계 쿼리 - 조직별 부서 수
result = db.run("SELECT o.org_name, COUNT(d.dept_id) as dept_count FROM organizations o LEFT JOIN departments d ON o.org_id = d.org_id GROUP BY o.org_name;")

print("조직별 부서 통계:")
print(result)

조직별 부서 통계:
[('건설안전교통국', 5), ('기획조정실', 4), ('농업환경국', 6), ('도시주택국', 6), ('문화체육국', 6), ('복지정책국', 5), ('부시장 직속', 3), ('전략산업국', 5), ('천안시청', 0), ('행정자치국', 5)]


In [109]:
# 조직별 부서 수와 층별 분포
query = """
SELECT
    o.org_name,
    COUNT(DISTINCT d.dept_id) as dept_count,
    COUNT(DISTINCT d.floor_location) as floor_count
FROM organizations o
LEFT JOIN departments d ON o.org_id = d.org_id
GROUP BY o.org_name
HAVING dept_count > 0
ORDER BY dept_count DESC;
"""

result = db.run(query)
print("조직별 부서 및 층 분포:")
print(result)

조직별 부서 및 층 분포:
[('문화체육국', 6, 4), ('도시주택국', 6, 4), ('농업환경국', 6, 2), ('행정자치국', 5, 3), ('전략산업국', 5, 4), ('복지정책국', 5, 2), ('건설안전교통국', 5, 2), ('기획조정실', 4, 3), ('부시장 직속', 3, 3)]


- ('문화체육국', 6, 4) : 문화체육국에 속한 6개 과는, 총 4개의 층에 분포해 있다.

### 6-3. 특정 부서의 건물 위치 조회

departments 테이블과 office_floors 테이블을 JOIN하여 특정 부서의 상세 위치 정보를 조회합니다.

In [110]:
# 테스트 1: "스마트정보과"의 건물 위치 조회
query = """
SELECT
    d.dept_name,
    d.phone,
    d.floor_location,
    f.building_name,
    f.floor_number,
    f.area_type,
    f.main_departments,
    f.facilities
FROM departments d
LEFT JOIN office_floors f ON d.floor_location = f.floor_name
WHERE d.dept_name = '스마트정보과';
"""

result = db.run(query)
print("스마트정보과의 위치 정보:")
print(result)
print("\n" + "=" * 80)

스마트정보과의 위치 정보:
[('스마트정보과', '041-521-2310', '본관 4층', '천안시청 본관', 4, '정보관리층', '스마트정보과,감사관', '복지정책국장실,전략산업국장실,감사실,방송실,통신실,전산실')]



In [111]:
# 테스트 2: LIKE 조건을 사용한 JOIN (여러 부서의 위치 정보 조회)
query = """
SELECT
    d.dept_name,
    d.phone,
    d.floor_location,
    f.building_name,
    f.floor_number,
    f.area_type,
    f.facilities
FROM departments d
LEFT JOIN office_floors f ON d.floor_location LIKE '%' || f.floor_name || '%'
WHERE d.dept_name IN ('스마트정보과', '민원여권과', '홍보담당관')
ORDER BY d.dept_name;
"""

result = db.run(query)
print("여러 부서의 위치 정보 (LIKE JOIN):")
print(result)
print("\n" + "=" * 80)

여러 부서의 위치 정보 (LIKE JOIN):
[('민원여권과', '041-521-2250', '본관 1층', '천안시청 본관', 1, '민원서비스층', '청사카페,민원실,대기공간'), ('스마트정보과', '041-521-2310', '본관 4층', '천안시청 본관', 4, '정보관리층', '복지정책국장실,전략산업국장실,감사실,방송실,통신실,전산실'), ('홍보담당관', '041-521-2080', '본관 8층', '천안시청 본관', 8, '기획정책층', '정책보좌관실,브리핑실,예산팀장실,기획조정실장실')]



---

### 참고 자료

- [Pandas to_sql 문서](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)
- [SQLite 공식 문서](https://www.sqlite.org/docs.html)
- [LangChain SQLDatabase](https://python.langchain.com/docs/integrations/tools/sql_database/)
- [SQLAlchemy 문서](https://docs.sqlalchemy.org/)